In [1]:
import pandas as pd
import time
import requests
import zipfile
import os
import json
import glob
import csv

In [2]:

base_url = "https://opentender.net/api/tender/export-ocds-batch/"
years = list(range(2015, 2024))

all_data = []
years

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

In [3]:
for year in years:
    target_url = f"{base_url}?year={year}&lpse=127"
    print(f"Downloading: Year {year}...")
    
    try:
        res = requests.get(target_url, stream=True)
        
        zip_filename = f"ocds_{year}.zip"
        extract_folder = f"data"
        
        with open(zip_filename, "wb") as f:
            for chunk in res.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        
        print(f"✅ Saved: {zip_filename}")
        
        # unzip
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)
        
        print(f"📂 Extracted to: {extract_folder}")
        
        os.remove(zip_filename)
        
        time.sleep(3)
        
    except Exception as e:
        print(f"❌ Error for {year}: {e}")

Downloading: Year 2015...
✅ Saved: ocds_2015.zip
📂 Extracted to: data
Downloading: Year 2016...
✅ Saved: ocds_2016.zip
📂 Extracted to: data
Downloading: Year 2017...
✅ Saved: ocds_2017.zip
📂 Extracted to: data
Downloading: Year 2018...
✅ Saved: ocds_2018.zip
📂 Extracted to: data
Downloading: Year 2019...
✅ Saved: ocds_2019.zip
📂 Extracted to: data
Downloading: Year 2020...
✅ Saved: ocds_2020.zip
📂 Extracted to: data
Downloading: Year 2021...
✅ Saved: ocds_2021.zip
📂 Extracted to: data
Downloading: Year 2022...
✅ Saved: ocds_2022.zip
📂 Extracted to: data
Downloading: Year 2023...
✅ Saved: ocds_2023.zip
📂 Extracted to: data


Convert json to CSV file

In [4]:
path_folder = 'data'
files = glob.glob(os.path.join(path_folder, "ocds-data-tender-*.json"))

output_file = 'All_data_ocd.csv'

fieldnames = [
    'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id',
    'tender_id', 'tender_title', 'mainProcurementCategory',
    'tender_minValue', 'tender_datePublished', 'tender_status',
    'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier'
]

with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    if not files:
        print("Folder kosong atau file tidak ditemukan!")
    else:
        total_rows = 0

        for file_path in files:
            print(f"Processing: {file_path}")

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                if 'releases' not in data:
                    continue

                for release in data['releases']:
                    tender = release.get('tender', {})
                    buyer = release.get('buyer', {})
                    awards = release.get('awards', [])

                    for award in awards:
                        suppliers = award.get('suppliers', [])
                        supplier_names = ', '.join([s.get('name', '') for s in suppliers])

                        writer.writerow({
                            'ocid': release.get('ocid'),
                            'release_id': release.get('id'),
                            'date': release.get('date'),
                            'buyer_name': buyer.get('name'),
                            'buyer_id': buyer.get('id'),
                            'tender_id': tender.get('id'),
                            'tender_title': tender.get('title'),
                            'mainProcurementCategory': tender.get('mainProcurementCategory'),
                            'tender_minValue': tender.get('minValue', {}).get('amount'),
                            'tender_datePublished': tender.get('datePublished'),
                            'tender_status': tender.get('status'),
                            'award_id': award.get('id'),
                            'award_title': award.get('title'),
                            'award_date': award.get('date'),
                            'award_value': award.get('value', {}).get('amount'),
                            'award_supplier': supplier_names
                        })

                        total_rows += 1

            except Exception as e:
                print(f"Gagal membaca {file_path}: {e}")

        print(f"Selesai! Total data: {total_rows} baris")

Processing: data\ocds-data-tender-02042026120121.json
Processing: data\ocds-data-tender-02042026120139.json
Processing: data\ocds-data-tender-02042026120148.json
Processing: data\ocds-data-tender-02042026120157.json
Processing: data\ocds-data-tender-02042026120217.json
Processing: data\ocds-data-tender-02042026120303.json
Processing: data\ocds-data-tender-02042026120314.json
Processing: data\ocds-data-tender-02042026120340.json
Processing: data\ocds-data-tender-02042026120359.json
Processing: data\ocds-data-tender-06042026130502.json
Processing: data\ocds-data-tender-06042026130508.json
Processing: data\ocds-data-tender-06042026130515.json
Processing: data\ocds-data-tender-06042026130523.json
Processing: data\ocds-data-tender-06042026130534.json
Processing: data\ocds-data-tender-06042026130541.json
Processing: data\ocds-data-tender-06042026130547.json
Processing: data\ocds-data-tender-06042026130554.json
Processing: data\ocds-data-tender-06042026130603.json
Processing: data\ocds-data-t